In [1]:
import pandas as pd
import numpy as np
import joblib

In [2]:
model_data = pd.read_csv("../data/processed/model_data.csv")

model = joblib.load("../models/world_cup_predictor_model.pkl")
features = joblib.load("../models/model_features.pkl")

In [3]:
model_data.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,result,result_label,home_recent_win_rate,home_recent_avg_goals_scored,home_recent_avg_goals_conceded,away_recent_win_rate,away_recent_avg_goals_scored,away_recent_avg_goals_conceded,win_rate_difference,attack_difference,defense_difference
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False,Draw,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False,Home Win,1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False,Home Win,1,0.000000,1.000000,2.000000,0.500000,2.000000,1.000000,-0.5,-1.000000,1.000000
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False,Draw,0,0.333333,1.666667,1.333333,0.333333,1.333333,1.666667,0.0,0.333333,-0.333333
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False,Home Win,1,0.250000,1.500000,1.750000,0.250000,1.750000,1.500000,0.0,-0.250000,0.250000


In [4]:
model_data["date"] = pd.to_datetime(model_data["date"])

In [5]:
home_team_features = model_data[[
    "date",
    "home_team",
    "home_recent_win_rate",
    "home_recent_avg_goals_scored",
    "home_recent_avg_goals_conceded"
]].copy()

home_team_features = home_team_features.rename(columns={
    "home_team": "team",
    "home_recent_win_rate": "recent_win_rate",
    "home_recent_avg_goals_scored": "recent_avg_goals_scored",
    "home_recent_avg_goals_conceded": "recent_avg_goals_conceded"
})

In [6]:
away_team_features = model_data[[
    "date",
    "away_team",
    "away_recent_win_rate",
    "away_recent_avg_goals_scored",
    "away_recent_avg_goals_conceded"
]].copy()

away_team_features = away_team_features.rename(columns={
    "away_team": "team",
    "away_recent_win_rate": "recent_win_rate",
    "away_recent_avg_goals_scored": "recent_avg_goals_scored",
    "away_recent_avg_goals_conceded": "recent_avg_goals_conceded"
})

In [7]:
team_features = pd.concat([home_team_features, away_team_features], ignore_index=True)

In [8]:
team_features = team_features.sort_values("date")

latest_team_features = team_features.groupby("team").tail(1).reset_index(drop=True)

latest_team_features.head()

,date,team,recent_win_rate,recent_avg_goals_scored,recent_avg_goals_conceded
0,1923-02-25,Asturias,0.0,0.0,0.0
1,1923-11-25,Central Spain,0.0,1.0,4.0
2,1942-08-09,Manchukuo,0.0,0.0,6.5
3,1956-06-06,Saarland,0.0,1.0,3.6
4,1970-09-20,North Vietnam,0.4,3.2,1.4


In [9]:
latest_team_features["team"].sort_values().unique()

array(['Abkhazia', 'Afghanistan', 'Albania', 'Alderney', 'Algeria',
       'Ambazonia', 'American Samoa', 'Andalusia', 'Andorra', 'Angola',
       'Anguilla', 'Antigua and Barbuda', 'Arameans Suryoye', 'Argentina',
       'Armenia', 'Artsakh', 'Aruba', 'Asturias', 'Australia', 'Austria',
       'Aymara', 'Azerbaijan', 'Bahamas', 'Bahrain', 'Bangladesh',
       'Barawa', 'Barbados', 'Basque Country', 'Belarus', 'Belgium',
       'Belize', 'Benin', 'Bermuda', 'Bhutan', 'Biafra', 'Bolivia',
       'Bonaire', 'Bosnia and Herzegovina', 'Botswana', 'Brazil',
       'British Virgin Islands', 'Brittany', 'Brunei', 'Bulgaria',
       'Burkina Faso', 'Burundi', 'Cambodia', 'Cameroon', 'Canada',
       'Canary Islands', 'Cape Verde', 'Cascadia', 'Catalonia',
       'Cayman Islands', 'Central African Republic', 'Central Spain',
       'Chad', 'Chagos Islands', 'Chameria', 'Chechnya', 'Chile', 'China',
       'Cilento', 'Colombia', 'Comoros', 'Congo', 'Cook Islands',
       'Corsica', 'Costa Rica',

In [10]:
def predict_match(home_team, away_team, neutral=True):
    # Check if home team exists
    if home_team not in latest_team_features["team"].values:
        return f"{home_team} not found in dataset."
    
    # Check if away team exists
    if away_team not in latest_team_features["team"].values:
        return f"{away_team} not found in dataset."
    
    # Get latest home team stats
    home_stats = latest_team_features[latest_team_features["team"] == home_team].iloc[0]
    
    # Get latest away team stats
    away_stats = latest_team_features[latest_team_features["team"] == away_team].iloc[0]
    
    # Create input row for model
    input_data = pd.DataFrame([{
        "home_recent_win_rate": home_stats["recent_win_rate"],
        "home_recent_avg_goals_scored": home_stats["recent_avg_goals_scored"],
        "home_recent_avg_goals_conceded": home_stats["recent_avg_goals_conceded"],
        "away_recent_win_rate": away_stats["recent_win_rate"],
        "away_recent_avg_goals_scored": away_stats["recent_avg_goals_scored"],
        "away_recent_avg_goals_conceded": away_stats["recent_avg_goals_conceded"],
        "win_rate_difference": home_stats["recent_win_rate"] - away_stats["recent_win_rate"],
        "attack_difference": home_stats["recent_avg_goals_scored"] - away_stats["recent_avg_goals_scored"],
        "defense_difference": home_stats["recent_avg_goals_conceded"] - away_stats["recent_avg_goals_conceded"],
        "neutral": neutral
    }])
    
    # Make prediction
    prediction = model.predict(input_data)[0]
    probabilities = model.predict_proba(input_data)[0]
    
    # Create readable labels
    label_map = {
        -1: "Away Win",
         0: "Draw",
         1: "Home Win"
    }
    
    probability_table = pd.DataFrame({
        "Result": [label_map[label] for label in model.classes_],
        "Probability": probabilities
    })
    
    probability_table["Probability"] = probability_table["Probability"].round(3)
    
    return {
        "home_team": home_team,
        "away_team": away_team,
        "predicted_result": label_map[prediction],
        "probabilities": probability_table
    }

In [11]:
predict_match("Brazil", "Argentina")

{'home_team': 'Brazil',
 'away_team': 'Argentina',
 'predicted_result': 'Home Win',
 'probabilities':      Result  Probability
 0  Away Win        0.385
 1      Draw        0.200
 2  Home Win        0.415}

In [12]:
predict_match("France", "England")

{'home_team': 'France',
 'away_team': 'England',
 'predicted_result': 'Draw',
 'probabilities':      Result  Probability
 0  Away Win        0.264
 1      Draw        0.377
 2  Home Win        0.359}

In [13]:
predict_match("Germany", "Spain")

{'home_team': 'Germany',
 'away_team': 'Spain',
 'predicted_result': 'Away Win',
 'probabilities':      Result  Probability
 0  Away Win         0.38
 1      Draw         0.33
 2  Home Win         0.29}

In [14]:
def show_prediction(home_team, away_team, neutral=True):
    result = predict_match(home_team, away_team, neutral)
    
    if isinstance(result, str):
        print(result)
        return
    
    print(f"Match: {result['home_team']} vs {result['away_team']}")
    print(f"Predicted Result: {result['predicted_result']}")
    print()
    print(result["probabilities"])

In [15]:
show_prediction("Brazil", "Argentina")

Match: Brazil vs Argentina
Predicted Result: Home Win

     Result  Probability
0  Away Win        0.385
1      Draw        0.200
2  Home Win        0.415
